In [ ]:
import pandas as pd
import numpy as np

listings = pd.read_csv('data/listings_clean.csv', parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])

print(listings.shape)
print(listings['price'].dtype, listings['last_review'].dtypes)

## Lab 6 - Part 2
### Ex 1

In [ ]:
for func in ['mean', 'median', 'std']:
    print(func, listings['review_scores_rating'].agg(func).round(2))

print(listings['review_scores_rating'].describe())

#### mean 4.82 median 4.92 std 0.32. the tail goes left, mean dragged down.
#### would use the median since the data trends towards 5. even 25% is 4.75.
### Ex 2

In [ ]:
scores = listings['review_scores_rating']

q1 = scores.quantile(0.25) 
q3 = scores.quantile(0.75) 
iqr = q3 - q1
upper = q3 + 1.5 * iqr 
lower = q1 - 1.5 * iqr

under_lower = scores < lower
over_lower = scores >= lower
under_5 = listings['number_of_reviews'] <= 5

print('iqr', iqr,
      '\nupper', upper,
      '\nlower', lower)
print(listings.loc[under_lower, 'review_scores_rating'].describe())
print(listings.loc[under_lower, 'number_of_reviews'].median(),
      listings.loc[over_lower, 'number_of_reviews'].median())

print('% under 5 reviews outliers',
      (listings.loc[under_lower & under_5, 'number_of_reviews'].count() /
      listings.loc[under_lower, 'number_of_reviews'].count()).round(4) * 100)
print('% under 5 reviews rest',
      (listings.loc[over_lower & under_5, 'number_of_reviews'].count() /
      listings.loc[over_lower, 'number_of_reviews'].count()).round(4) * 100)

print(listings[under_lower].groupby('room_type')['number_of_reviews'].count())

listings['rating_outlier'] = under_lower

#### iqr 0.25 upper 5.375 lower 4.375
#### 211 listings under lower fence. median reviews under lower fence: 4. 13 for the others.
#### Under 5 reviews: outliers: 59.24%. rest: 28.13%
#### outlier room_type: Entire home/apt    160 Private room        51
#### Decided to keep outliers and add column rating_ouliers bool. Fill.
### Ex 3


In [ ]:
# a)
is_super = listings['host_is_superhost']

for var in ['is super', 'not super']:
    for func in ['count', 'median']:
        scope = is_super if var=='is super' else ~is_super
        print(var, func, listings[scope]['number_of_reviews'].agg(func))

# b)
listings['years_old'] = (listings['last_scraped'] - listings['first_review']).dt.days / 365.25

print(listings.groupby('host_is_superhost')['years_old'].median())
print('superhost review multi', 53/7, 'year delta', 3.31-2.89)

# c)
for var in ['is super', 'not super']:
    for func in ['count', 'median']:
        scope = is_super if var=='is super' else ~is_super
        print(var, func, listings[scope]['number_of_reviews_ltm'].agg(func))

#### (a) superhosts: 737 listings, median 53. Others: 3837 listings, median 7.
#### (b) Years: 3.31 for superhosts and 2.89 for the others.
#### that is only about 0.42 years longer. not large enough difference to explain the x7.5 amount of reviews superhosts have.
#### (c) we can see that within the last year superhosts have gotten 16x as many reviews as non-superhosts. 
#### so we can conclude that superhosts get much larger amount of reviews. we can not conclude exactly why this is.
#### could be better hosts / quality of recsidence. could be gaming the system adding fake reviews, dropping accounts with bad rating for a fresh.
### Ex 4

- **number_of_reviews:** although they scew the data they are real values and should be kept we decide to keep and mark them with a bool column `reviews_outlier` count: `582`. Fill.
- **review_scores_rating:** although they scew the data they are real values and should be kept we decide to keep and mark them with a bool column `review_outlier` count: `211` . Fill.